# Lab 02 — Train / Validation / Test Split (and Avoiding Leakage)

**Goal:** practice splitting datasets correctly, see what data leakage looks like in code, and learn cross-validation as a way to use the data more efficiently.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

SEED = 42

## 2. A correct 70 / 15 / 15 split

In [2]:
df = fetch_california_housing(as_frame=True).frame
X = df.drop(columns=['MedHouseVal']).values
y = df['MedHouseVal'].values

# First carve off the test set (15%).
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)
# Then split the rest into train (~70% of total) and val (~15% of total).
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15 / 0.85, random_state=SEED
)

print(f'train: {X_train.shape[0]} samples ({X_train.shape[0] / len(X):.0%})')
print(f'val  : {X_val.shape[0]} samples ({X_val.shape[0]   / len(X):.0%})')
print(f'test : {X_test.shape[0]} samples ({X_test.shape[0]  / len(X):.0%})')

train: 14448 samples (70%)
val  : 3096 samples (15%)
test : 3096 samples (15%)


## 3. Stratified split for classification

In [3]:
iris = load_iris(as_frame=True).frame
Xc = iris.drop(columns=['target']).values
yc = iris['target'].values

_, _, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=SEED, stratify=yc)
print('Class balance in test split (stratified):')
print(pd.Series(yte).value_counts(normalize=True).round(3))

_, _, ytr_u, yte_u = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)
print('\nClass balance in test split (unstratified):')
print(pd.Series(yte_u).value_counts(normalize=True).round(3))

Class balance in test split (stratified):
0    0.333
2    0.333
1    0.333
Name: proportion, dtype: float64

Class balance in test split (unstratified):
2    0.367
0    0.333
1    0.300
Name: proportion, dtype: float64


Stratification keeps the class proportions the same across the splits. Use it for any imbalanced classification problem.

## 4. Data leakage — the wrong way

Below: fit a StandardScaler on the **entire** dataset before splitting. The test-set statistics leak into training. The model can score artificially well during development, then fail in production.

In [4]:
# WRONG: scaler fit on the whole X (test included)
scaler_wrong = StandardScaler().fit(X)
X_scaled = scaler_wrong.transform(X)
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED
)
model_w = LinearRegression().fit(X_train_w, y_train_w)
rmse_w = float(np.sqrt(mean_squared_error(y_test_w, model_w.predict(X_test_w))))
print(f'WRONG (leaked) test RMSE = {rmse_w:.4f}')

WRONG (leaked) test RMSE = 0.7456


## 5. The right way

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
scaler_right = StandardScaler().fit(X_tr)        # fit on train only
X_tr_s = scaler_right.transform(X_tr)
X_te_s = scaler_right.transform(X_te)
model_r = LinearRegression().fit(X_tr_s, y_tr)
rmse_r  = float(np.sqrt(mean_squared_error(y_te, model_r.predict(X_te_s))))
print(f'RIGHT       test RMSE = {rmse_r:.4f}')

RIGHT       test RMSE = 0.7456


For this dataset the two numbers are very close because California Housing is large and homogeneous. In smaller or more skewed datasets the gap can be huge — and always points the wrong way (leakage looks too good).

## 6. K-fold cross-validation

Validation can waste data. K-fold CV uses all the training data across $K$ rotations: each row is used for validation exactly once.

In [6]:
scaler_cv = StandardScaler().fit(X_tr)
X_tr_s_cv = scaler_cv.transform(X_tr)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rmses = []
for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_tr_s_cv), start=1):
    m = LinearRegression().fit(X_tr_s_cv[tr_idx], y_tr[tr_idx])
    pred = m.predict(X_tr_s_cv[va_idx])
    rmse = float(np.sqrt(mean_squared_error(y_tr[va_idx], pred)))
    cv_rmses.append(rmse)
    print(f'fold {fold_idx}: RMSE = {rmse:.4f}')

print(f'\nmean CV RMSE = {np.mean(cv_rmses):.4f} (± {np.std(cv_rmses):.4f})')

fold 1: RMSE = 0.7339
fold 2: RMSE = 0.7252
fold 3: RMSE = 0.6973
fold 4: RMSE = 0.7334
fold 5: RMSE = 0.7128

mean CV RMSE = 0.7205 (± 0.0139)


Reporting **mean ± std** across folds gives a much more honest sense of model performance than a single train/val split.

## 7. Summary

- Carve off the test set first. Touch it only once.
- Use `stratify=` for imbalanced classification.
- Always fit preprocessing on train, then transform val/test.
- K-fold CV gives mean ± std performance, which is what you should report.
- The next chapters add real models inside this scaffolding.